# Customer Churn Analysis and Prediction Using Machine Learning

**Author:** Vinay Sancheti  
**Internship:** IBM SkillsBuild Data Analytics with AI Academic Internship Program

## Project objective
Analyze telecom customer churn, identify important churn patterns, and build machine-learning classification models to predict whether a customer is likely to churn.

> Run all cells from top to bottom. The notebook will download the IBM sample CSV if it is not already available locally.

In [ ]:
# Import required libraries
from pathlib import Path
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load the dataset

The project uses the IBM Telco Customer Churn sample dataset. If the dataset is not found in the local `data` directory, the code attempts to download it from IBM's archived GitHub repository.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "Telco-Customer-Churn.csv"
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

if not DATA_PATH.exists():
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
        print("Dataset downloaded successfully.")
    except Exception as exc:
        print("Automatic download failed.")
        print("Please download the dataset manually from:")
        print("https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv")
        raise exc

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()

## 2. Understand the data

Inspect data types, summary statistics, duplicate records, and missing values before modeling.

In [ ]:
print("Rows and columns:", df.shape)
display(df.head())
display(df.info())

In [ ]:
# Summary statistics
display(df.describe(include="all").T)

In [ ]:
# Missing values
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("Missing Values"))

# Duplicate customer IDs
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

## 3. Data cleaning

`TotalCharges` can contain blank strings in the source data. Convert it to numeric and impute missing values using the median. The customer ID is an identifier and is excluded from predictive modeling.

In [ ]:
df = df.copy()

# Convert TotalCharges to numeric; invalid/blank values become NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Fill missing TotalCharges with the median
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Remove duplicate customer records if any
df = df.drop_duplicates(subset="customerID").reset_index(drop=True)

# Binary target
df["ChurnFlag"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Cleaned shape:", df.shape)
print("Remaining missing values:", int(df.isna().sum().sum()))

## 4. Exploratory Data Analysis

The following visualizations examine churn distribution and relationships between churn and important customer attributes.

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

print("Overall churn rate: {:.2f}%".format(df["ChurnFlag"].mean() * 100))

In [ ]:
def churn_rate_table(column):
    result = (
        df.groupby(column)["ChurnFlag"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "churn_rate"})
        .sort_values("churn_rate", ascending=False)
    )
    result["churn_rate"] = result["churn_rate"] * 100
    return result

contract_churn = churn_rate_table("Contract")
display(contract_churn)

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="Contract", y="ChurnFlag", errorbar=None)
plt.title("Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.xlabel("Contract")
plt.show()

In [ ]:
payment_churn = churn_rate_table("PaymentMethod")
display(payment_churn)

plt.figure(figsize=(10, 5))
sns.barplot(data=df, x="PaymentMethod", y="ChurnFlag", errorbar=None)
plt.title("Churn Rate by Payment Method")
plt.ylabel("Churn Rate")
plt.xlabel("Payment Method")
plt.xticks(rotation=25, ha="right")
plt.show()

In [ ]:
internet_churn = churn_rate_table("InternetService")
display(internet_churn)

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="InternetService", y="ChurnFlag", errorbar=None)
plt.title("Churn Rate by Internet Service")
plt.ylabel("Churn Rate")
plt.xlabel("Internet Service")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure Distribution by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Tenure (Months)")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
corr = df[numeric_cols + ["ChurnFlag"]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix for Numeric Variables")
plt.show()

## 5. Prepare data for machine learning

Categorical features are one-hot encoded and numerical features are standardized. The preprocessing is placed inside a scikit-learn pipeline to avoid data leakage.

In [ ]:
X = df.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = df["ChurnFlag"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 6. Train classification models

Three commonly used classification algorithms are compared:

- Logistic Regression
- Decision Tree
- Random Forest

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=10, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=4,
        random_state=42, n_jobs=-1
    )
}

pipelines = {
    name: Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}

results = []
fitted_models = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
display(results_df)

## 7. Classification reports and confusion matrices

In [ ]:
for name, pipe in fitted_models.items():
    y_pred = pipe.predict(X_test)

    print("=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(
        y_test, y_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["No Churn", "Churn"],
        yticklabels=["No Churn", "Churn"]
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## 8. ROC curve comparison

In [ ]:
plt.figure(figsize=(8, 6))

for name, pipe in fitted_models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=plt.gca())

plt.title("ROC Curves for Churn Classification Models")
plt.show()

## 9. Feature importance / model interpretation

For the Random Forest model, the transformed feature names are extracted and ranked by importance. This provides an interpretable view of which variables the trained model relied on most.

In [ ]:
rf_pipeline = fitted_models["Random Forest"]

feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()
rf_model = rf_pipeline.named_steps["model"]

importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(importance.head(15).to_frame("Importance"))

plt.figure(figsize=(9, 6))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 10. Final observations

The analysis should be interpreted together with the model metrics rather than relying on a single visualization. In particular, contract type, tenure, service configuration, payment method, and monthly charges are useful dimensions for examining churn.

The dataset is a cross-sectional sample, so the model should be treated as an educational classification exercise rather than a guarantee of future customer behavior.

In [ ]:
print("Final model comparison:")
display(results_df)

print("\nHighest observed churn-rate contract segment:")
display(contract_churn.head(1))

print("\nHighest observed churn-rate payment segment:")
display(payment_churn.head(1))

print("\nHighest observed churn-rate internet-service segment:")
display(internet_churn.head(1))

## 11. Conclusion

This project demonstrates an end-to-end data analytics and machine-learning workflow for customer churn. It combines data cleaning, exploratory analysis, visualization, feature preprocessing, supervised learning, evaluation, and interpretation.

The notebook is designed so that all numerical model results are generated directly from the source dataset when the notebook is executed.